# Dashboard evidence preparation and release validation

This notebook prepares local aggregate inputs, not a published website. Source data remain read-only. It consumes the same-source completed rule-validation run and exports no transaction rows, accounts, BIC identifiers, timestamps, detail strings, or literal tokens. Implementation: `dashboard_prep` in `src/remaining_analysis.py`.

All outputs are versioned. Aggregate CSVs stay under ignored outputs; a local dashboard connection manifest points to them. Competition material is not committed, pushed, uploaded, or published. Aggregate information may still be competition-restricted; local preparation is not authorization for public release.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / ".venv/Scripts/python.exe").exists()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), "Use the project virtual environment"
sys.path.insert(0, str(ROOT / "src"))
from remaining_analysis import Study, dashboard_prep
study = Study(ROOT, '09_dashboard_data_prep', prerequisites=('05_currency_party', '06_anomaly_analysis', '07_tokens', '08_rule_validation'))

## Overview and dashboard table grains

**Purpose.** Make the evidence usable while preserving exact population reconciliation.

**Hypothesis.** Independent aggregate views reconcile to 1,048,575 rows and preserve duration in seconds and label categories.

**Methodology.** Export overview, route, party, currency and label profiles; a route×currency×party cube with additive counts and sums; fixed duration-band histograms; UTC daily totals; the scoped rule register; and a data dictionary. Medians and percentiles are computed directly at each displayed grain and must never be summed or averaged across filters. Means and rates must be recomputed from appropriate sums and denominators. Separate tables are alternative views and must not be unioned into one population.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Matching counts and additive totals validate extraction. Dashboard statistics remain descriptive; UNLABELED retains its original uncertainty and CUSTOMER_RECALL remains separate.

**Limitations.** A cube cannot recover filtered medians from cell medians. UTC date is not local operating time. No FX exchange rate or fee is observed.

**Business Implications.** Provide reproducible segment benchmarking and rule evidence without invented savings or causal performance claims.

In [ ]:
overview, cube, histogram, daily, privacy, rule_table = dashboard_prep(study)
display(overview)
display(cube.head(20))
display(histogram)
display(daily.head(10))

## Small-cell accounting and output validation

**Purpose.** Check that aggregate preparation neither loses transactions nor exposes direct identifiers.

**Hypothesis.** No output contains forbidden identifier columns or transaction-sized row sets; small cube cells can be rolled up with exact count reconciliation.

**Methodology.** Cells with n<20 in the multifactor cube are represented by one SUPPRESSED_SMALL_CELLS row containing only additive totals; its median/P95 are deliberately undefined. Preserve its count in overall totals and report how many rows cannot be attributed after filtering. Check count, label count, amount sum and duration sum reconciliation, plus an explicit identifier-column denylist. Source SHA-256 is checked again before completion.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** These checks establish local output structure, not formal anonymization or permission to publish. Fine-grained external release needs a separate competition-policy review.

**Limitations.** Small-cell rollup restricts filtered attribution. Privacy checks are structural, not a differential-privacy guarantee. Amount sums use floating-point arithmetic with stated tolerance.

**Business Implications.** Avoid misleading filtered totals and keep competition data within the local project.

In [ ]:
display(privacy)
display(rule_table[["rule_id","condition","status","confidence"]])
display(pd.read_csv(study.tables / "dashboard_data_dictionary.csv"))
display(Image(filename=str(study.charts / "dashboard_route_duration.png")))

In [ ]:
report = study.finish([f"Dashboard population = {int(overview.n.iloc[0]):,}; median duration = {overview.median_duration_seconds.iloc[0]:g} seconds; observed labeled rows = {int(overview.labeled_n.iloc[0]):,}.",
                       f"Cube rows = {len(cube):,}; small-cell rollup covers {int(privacy.suppressed_transactions.iloc[0]):,} transactions from {int(privacy.suppressed_original_cells.iloc[0]):,} original cells.",
                       f"Duration histogram reconciles to {int(histogram.n.sum()):,}; UTC daily totals reconcile to {int(daily.n.sum()):,}.",
                       f"Scoped rule evidence rows = {len(rule_table)}; no transaction-level exports."], ['Dashboard medians/percentiles are nonadditive; only exact-grain values may be displayed.', 'Suppressed cube cells cannot be attributed after route/currency/party filtering.', 'Local aggregate preparation is not public-release authorization; no publishing was performed.', 'Earlier exploratory candidate questions remain open in the reconciliation table.'], 'Use the aggregate manifest for a local dashboard, and verify organizer definitions and unvalidated bank/time hypotheses before submission claims.')
display(Markdown("### Final findings and limitations\n" + "\n".join("- " + x for x in report["findings"]) + "\n\n" + " ".join(report["limitations"])))
display(pd.DataFrame(study.checks))
display(Markdown("### Completion summary\nExecution passed for " + format(report["population"], ",") + " transactions. All generated files are listed below. No unresolved material errors.\n\n" + "\n".join("- `"+x+"`" for x in report["files"]) + "\n\nRecommended next investigation: " + report["next_investigation"]))